In [1]:
import pandas as pd
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv('../data/processed/rq1_features.csv', index_col='Datetime (UTC)', parse_dates=True)

In [3]:
price_features = [
    'price_lag_1h', 'price_lag_24h', 'price_lag_168h',
    'hour', 'day_of_week', 'month', 'is_weekend',
    'temperature_2m', 'wind_speed_10m', 'cloud_cover', 'shortwave_radiation',
    'P', 'P_lag_1h', 'P_lag_24h'
]

In [4]:
X_price = df[price_features]
y_price = df['Price (EUR/MWhe)']

In [5]:
train_end = int(len(df) * 0.70)
validation_end = int(len(df) * 0.85)

X_train, y_train = X_price.iloc[:train_end], y_price.iloc[:train_end]
X_val, y_val     = X_price.iloc[train_end:validation_end], y_price.iloc[train_end:validation_end]

In [6]:
price_model = XGBRegressor(
    n_estimators=1000, max_depth=3, learning_rate=0.05,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1, reg_lambda=1, random_state=42,
    early_stopping_rounds=20
)
price_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

df['predicted_price'] = price_model.predict(X_price)

In [7]:
df['predicted_P'] = df['P_lag_24h']

In [8]:
df_rq3 = df.iloc[validation_end:].copy()

df_rq3 = df_rq3[[
    'Price (EUR/MWhe)', 'predicted_price',
    'P', 'predicted_P',
    'cloud_cover', 'is_weekend'
]].rename(columns={
    'Price (EUR/MWhe)': 'actual_price',
    'P': 'actual_P'
})

print(df_rq3.head())
print(f"Вкупно часови за RQ3 симулација: {len(df_rq3)}")

                     actual_price  predicted_price  actual_P  predicted_P  \
Datetime (UTC)                                                              
2023-09-18 06:00:00        121.28       130.585495     36.78        45.18   
2023-09-18 07:00:00        105.42       114.311111     71.63        84.68   
2023-09-18 08:00:00         91.32       101.201805    119.81       109.23   
2023-09-18 09:00:00         77.03        81.775986    165.83       122.79   
2023-09-18 10:00:00         67.02        66.665848    216.48       136.69   

                     cloud_cover  is_weekend  
Datetime (UTC)                                
2023-09-18 06:00:00          100           0  
2023-09-18 07:00:00          100           0  
2023-09-18 08:00:00          100           0  
2023-09-18 09:00:00          100           0  
2023-09-18 10:00:00          100           0  
Вкупно часови за RQ3 симулација: 306


In [9]:
df_rq3.to_csv('../data/processed/rq3_inputs.csv')